<a href="https://colab.research.google.com/github/sumaramuskan/Fake-News-Detection/blob/main/Fake_News_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -q kagglehub
import kagglehub

# Download dataset
path = kagglehub.dataset_download("algord/fake-news")
print("Dataset path:", path)

100%|██████████| 1.68M/1.68M [00:00<00:00, 2.99MB/s]

Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/algord/fake-news/versions/1


In [ ]:
import pandas as pd
import os

# Load dataset
df = pd.read_csv(os.path.join(path, "FakeNewsNet.csv"))
df.head()


,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [ ]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23196 entries, 0 to 23195
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          23196 non-null  object
 1   news_url       22866 non-null  object
 2   source_domain  22866 non-null  object
 3   tweet_num      23196 non-null  int64 
 4   real           23196 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 906.2+ KB


,0
title,0
news_url,330
source_domain,330
tweet_num,0
real,0


In [ ]:
df["real"].value_counts()

,count
real,
1,17441
0,5755


In [ ]:
df = df[["title", "real"]]
df.columns = ["text", "label"]

In [ ]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1,2)
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = lr_model.predict(X_test_tfidf)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 584  567]
 [ 145 3344]]
              precision    recall  f1-score   support

           0       0.80      0.51      0.62      1151
           1       0.86      0.96      0.90      3489

    accuracy                           0.85      4640
   macro avg       0.83      0.73      0.76      4640
weighted avg       0.84      0.85      0.83      4640



In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
lr_model.fit(X_train_tfidf, y_train)
y_pred = lr_model.predict(X_test_tfidf)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 871  280]
 [ 597 2892]]
              precision    recall  f1-score   support

           0       0.59      0.76      0.67      1151
           1       0.91      0.83      0.87      3489

    accuracy                           0.81      4640
   macro avg       0.75      0.79      0.77      4640
weighted avg       0.83      0.81      0.82      4640



In [ ]:
def predict_news(text):
    clean = clean_text(text)
    vec = vectorizer.transform([clean])
    return "REAL NEWS" if lr_model.predict(vec)[0] == 1 else "FAKE NEWS"
predict_news("Government confirms new economic policy after cabinet meeting")

'REAL NEWS'

In [ ]:
predict_news("Shocking truth the media doesn't want you to know!!!")

'FAKE NEWS'

In [ ]:
pip install requests


In [ ]:
import requests

API_KEY = "8a80bfd0-747b-4ae2-b332-d8de4fea2735"

url = f"https://content.guardianapis.com/search?api-key={API_KEY}&show-fields=headline,bodyText"

response = requests.get(url)
data = response.json()

articles = data['response']['results']


for article in articles:
    print("Title:", article['webTitle'])
    print("URL:", article['webUrl'])
    print("-" * 50)

Title: ‘One of America’s greatest patriots’: US political leaders pay tribute to Jesse Jackson
URL: https://www.theguardian.com/us-news/2026/feb/17/jesse-jackson-death-tribute
--------------------------------------------------
Title: Jesse Jackson: tributes and reactions from Bernice King, Trump and Biden after civil rights leader’s death – latest updates
URL: https://www.theguardian.com/us-news/live/2026/feb/17/jesse-jackson-tributes-died-84-civil-rights-latest-news-updates
--------------------------------------------------
Title: Farage insults female reporter as Braverman says Reform UK wants to scrap Equality Act – UK politics live
URL: https://www.theguardian.com/politics/live/2026/feb/17/reform-farage-shadow-cabinet-local-council-elections-labour-starmer-latest-news-updates
--------------------------------------------------
Title: Jim Ratcliffe’s repugnant words have sullied Manchester United’s reputation | Letters
URL: https://www.theguardian.com/uk-news/2026/feb/17/jim-ratcliff

In [ ]:
print(len(articles))

10


In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score())